In [1]:
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting

!pip install plyfile tqdm opencv-python joblib
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

from simple_knn._C import distCUDA2
print("✅ All installed correctly!")

Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 35.48 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (17

ImportError: libc10.so: cannot open shared object file: No such file or directory

In [1]:
import os, torch

# Fix missing libc10.so
torch_lib_path = os.path.join(os.path.dirname(torch.__file__), 'lib')
os.environ['LD_LIBRARY_PATH'] = torch_lib_path + ':' + os.environ.get('LD_LIBRARY_PATH', '')
!ldconfig

# Also fix simple_knn FLT_MAX error proactively
!sed -i '1s/^/#include <float.h>\n/' submodules/simple-knn/simple_knn.cu
!pip install submodules/simple-knn

# Verify
from simple_knn._C import distCUDA2
print("✅ simple_knn working!")

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create dataset folder structure
os.makedirs('/content/my_dataset/images', exist_ok=True)
os.makedirs('/content/my_dataset/sparse/0', exist_ok=True)

# Copy images from Drive
!cp -r /content/drive/MyDrive/playroom/images/* /content/my_dataset/images/

# Copy the 3 critical COLMAP files
!cp /content/drive/MyDrive/playroom/sparse/0/cameras.bin /content/my_dataset/sparse/0/
!cp /content/drive/MyDrive/playroom/sparse/0/images.bin /content/my_dataset/sparse/0/
!cp /content/drive/MyDrive/playroom/sparse/0/points3D.bin /content/my_dataset/sparse/0/

# Verify everything
img_count = len(os.listdir('/content/my_dataset/images'))
print(f"✅ Images copied: {img_count}")

for f in ['cameras.bin', 'images.bin', 'points3D.bin']:
    path = f'/content/my_dataset/sparse/0/{f}'
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {f}")

Mounted at /content/drive
✅ Images copied: 225
✅ cameras.bin
✅ images.bin
✅ points3D.bin


In [3]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

%cd /content/gaussian-splatting

!python train.py \
    -s /content/my_dataset \
    -m /content/output \
    --iterations 30000 \
    --resolution 4 \
    --densify_grad_threshold 0.0003 \
    --densification_interval 200 \
    --densify_until_iter 12000 \
    --save_iterations 7000 15000 30000 \
    --checkpoint_iterations 7000 15000 30000

/content/gaussian-splatting
2026-04-27 07:20:32.188890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777274432.209945   11046 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777274432.217492   11046 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777274432.237749   11046 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777274432.237798   11046 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777274432.237804   11046 computation_placer.cc

In [4]:
!cp -r /content/output /content/drive/MyDrive/gaussian_splatting_output
print("✅ Saved to Drive!")

✅ Saved to Drive!


In [5]:
%cd /content/gaussian-splatting
!python render.py -m /content/output
!cp -r /content/output/train /content/drive/MyDrive/gaussian_splatting_output/
!cp -r /content/output/test /content/drive/MyDrive/gaussian_splatting_output/
print("✅ Renders saved to Drive!")

/content/gaussian-splatting
Looking for config file in /content/output/cfg_args
Config file found: /content/output/cfg_args
Rendering /content/output
Loading trained model at iteration 30000 [27/04 07:32:45]
Reading camera 225/225 [27/04 07:32:46]
Loading Training Cameras [27/04 07:32:46]
Loading Test Cameras [27/04 07:32:51]
Rendering progress: 100% 225/225 [00:14<00:00, 15.17it/s]
Rendering progress: 0it [00:00, ?it/s]
✅ Renders saved to Drive!
